#### **FEATURE ENGINEERING**

In [43]:
import pandas as pd 
import numpy as np

catalogo_productos = pd.read_csv("Datos-finales/catalogo_productos.csv")
especificaciones_cajas = pd.read_csv("Datos-finales/especificaciones_cajas.csv")
operaciones_planta = pd.read_csv("Datos-finales/operaciones_planta.csv")
procurement_cajas = pd.read_csv("Datos-finales/procurement_cajas.csv")

Necesitamos averiguar el volumen total de cada producto, que nos servirá para luego calcular el porcentaje de la utilización de la caja nueva (y actual).

In [44]:
df_merge = catalogo_productos.merge(especificaciones_cajas,
                                    on='caja_tipo_id',
                                    how='left')

# Calculamos el volumen del producto como la dimensión interna de la caja actual, pues la 
# restricción indica que su valor es el mínimo posible para contener el producto.
df_merge['dim_producto_volumen'] = (df_merge['caja_interior_largo'] * 
                                    df_merge['caja_interior_ancho'] * 
                                    df_merge['caja_interior_alto'])

catalogo_productos['dim_producto_largo'] = df_merge['caja_interior_largo']
catalogo_productos['dim_producto_ancho'] = df_merge['caja_interior_ancho']
catalogo_productos['dim_producto_alto'] = df_merge['caja_interior_alto']
catalogo_productos['dim_producto_volumen'] = df_merge['dim_producto_volumen']

Además, agregamos dos columnas extra que indican la dimensión interna de cada tipo de caja.

In [45]:
especificaciones_cajas['dimension_interna'] = (
    especificaciones_cajas['caja_interior_ancho'] *
    especificaciones_cajas['caja_interior_largo'] *
    especificaciones_cajas['caja_interior_alto']
) 

Agreguemos en el archivo de compras unas columnas que indican por planta el volumen de tipo de planta que se necesitan para cumplir con las unidades de producto. Ya chequeamos anteriormente que hay más tipos de cajas comprados que los que se usan.

In [46]:
prod_op_merge = pd.concat([catalogo_productos, operaciones_planta], axis=1)

plantas = ['buenos_aires', 'curitiba', 'santiago', 'monterrey', 'bakersfield']
cols = [f'volumen_producto_planta_{p}' for p in plantas]

# Agrupamos los volúmenes de productos según cada planta por tipo de caja
volumen_por_caja = (prod_op_merge.groupby('caja_tipo_id')[cols].sum().reset_index())

# Renombramos las columnas para mejor control
volumen_por_caja.columns = ['caja_tipo_id'] + [f'volumen_tipo_planta_{p}_requerido' for p in plantas]
procurement_cajas = procurement_cajas.merge(volumen_por_caja, on='caja_tipo_id')

Eliminamos las columnas que no aporten valor al contexto del problema.

In [47]:
catalogo_productos.drop([
    'descripcion_producto', 'ingrediente_forma', 'tipo_proyecto', 'subcategoria',
    'categoria', 'tamaño_corte', 'tamaño_paquete'
], axis=1, inplace=True)

operaciones_planta.drop([
    'volumen_producto_canal_servicios_comida', 'volumen_producto_canal_minorista', 
    'volumen_producto_canal_cadenas_corporativas', 'volumen_producto_canal_otros'
], axis=1, inplace=True)

Exportamos las tablas modificadas en la carpeta "Datos-finales"

In [48]:
catalogo_productos.to_csv("Datos-finales/catalogo_productos.csv", index=False)
especificaciones_cajas.to_csv("Datos-finales/especificaciones_cajas.csv", index=False)
operaciones_planta.to_csv("Datos-finales/operaciones_planta.csv", index=False)
procurement_cajas.to_csv("Datos-finales/procurement_cajas.csv", index=False)